# AutoML-Gewinner verifizieren

**Autor:** qmowo · **Datum:** 2026-10-29

**Ziel:** Die eingefrorene kombinierte RBF-SVM ohne neues Tuning überprüfen, mit dem aktuellen Code reproduzieren und die Scores aus `origin/exp/automl-optuna` richtig einordnen.

Die Prüfung unterscheidet historische Inferenz, identische Daten/Splits und einen frischen Fit mit fest vorgegebenen Parametern. Sie ist **kein neuer unabhängiger Abschlusstest**. Der gesicherte Holdout wurde bereits betrachtet. Das Modellpaket wird nicht verändert; gemeinsame Datenartefakte werden nicht neu erzeugt.

## 1. Umgebung und gesicherte Referenz

Ausgangspunkt ist das exportierte Paket `models/automl_final`, nicht eine aus dem Gedächtnis übernommene Gewinnerkonfiguration.

In [ ]:
import importlib.metadata
import json
import platform
import subprocess
import sys
from pathlib import Path

import pandas as pd

from awp2.config import MODELS_DIR, PROJECT_ROOT

bundle = MODELS_DIR / "automl_final"
assert bundle.is_dir(), f"Gesichertes Modellpaket fehlt: {bundle}"
bundle_metadata = json.loads((bundle / "bundle.json").read_text(encoding="utf-8"))
frozen_record = json.loads((bundle / "provenance" / "validation.json").read_text(encoding="utf-8"))
assert frozen_record["status"] == "completed"
assert frozen_record["run_id"] == bundle_metadata["run_id"]
git_head = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
display(
    pd.Series(
        {
            "Paket": str(bundle.relative_to(PROJECT_ROOT)),
            "Run-ID": bundle_metadata["run_id"],
            "Modell-ID": bundle_metadata["model_id"],
            "Git-HEAD aktuell": git_head,
            "Python aktuell / Export": f"{platform.python_version()} / {bundle_metadata['python']}",
            "Plattform": platform.platform(),
            "Historische Holdout-BAcc kombiniert": bundle_metadata["metrics"]["bacc_combined"],
        },
        name="Gesicherte Referenz",
    )
)
display(
    pd.DataFrame(
        {
            "Paket": bundle_metadata["runtime_packages"],
            "Aktuell": {
                name: importlib.metadata.version(name)
                for name in bundle_metadata["runtime_packages"]
            },
        }
    )
)

## 2. Historischen Wert unabhängig vom aktuellen Code prüfen

Das mitgelieferte Prüfwerkzeug wird in einem isolierten Python-Prozess ausgeführt. Es prüft die Datei-Hashes, lädt ausschließlich den Paket-Code und vergleicht **alle Referenzvorhersagen**, nicht nur die gerundete BAcc. Die lokale MLflow-Datenbank wird dafür nicht benötigt. Das Paket enthält vertrauenswürdiges Cloudpickle; keine fremden Modellpakete ungeprüft laden.

Der folgende Schritt ist **nur Inferenz**. Er trainiert nichts und verändert das Paket nicht.

In [ ]:
import os
import tempfile

with tempfile.TemporaryDirectory(prefix="awp2-notebook-verification-") as isolated:
    environment = dict(os.environ)
    environment["MLFLOW_TRACKING_URI"] = f"sqlite:///{Path(isolated).as_posix()}/unused.sqlite"
    frozen_process = subprocess.run(
        [
            sys.executable,
            "-I",
            "-B",
            str(bundle / "model_bundle.py"),
            "verify",
            str(bundle),
        ],
        cwd=isolated,
        env=environment,
        capture_output=True,
        text=True,
        check=False,
    )
    if frozen_process.returncode:
        raise RuntimeError(frozen_process.stderr + frozen_process.stdout)
    frozen_verification = json.loads(frozen_process.stdout)

assert frozen_verification["status"] == "passed"
assert frozen_verification["identical_predictions"]
assert frozen_verification["bundled_code"]
assert frozen_verification["metrics"] == frozen_record["metrics"]
display(pd.Series(frozen_verification["metrics"], name="Historisch reproduziert"))
print(f"Alle {frozen_verification['rows']} historischen Vorhersagen stimmen exakt überein.")

## 3. Daten, Splits und mögliche Lecks prüfen

Es gelten nur die gemeinsamen Artefakte von `load_split()` / `load_folds()`. Ein gleicher 80/20-Anteil allein genügt nicht: Datei-Hashes, Sample-IDs, Feature-Reihenfolge und sämtliche Holdout-Werte müssen zur Referenz passen. Bei einer Abweichung stoppt das Notebook vor dem Fit.

Die AEZ×Monat-Überlappung wird **berichtet**, nicht versteckt: Zufällige Folds prüfen andere Messzeilen, nicht notwendigerweise bislang ungesehene Aufnahmen/Regionen. Ein überlappender Kontext ist nicht automatisch ein Zielvariablen-Leck, begrenzt aber die Generalisierungsaussage.

In [ ]:
import hashlib

from awp2.config import AEZ_COL, AEZ_MONTH_SEP, MONTH_COL, SEED, TARGET_COLS
from awp2.data import combined_label, load_folds, load_split, load_split_config

search_manifest = json.loads(
    (bundle / "provenance" / "search_manifest.json").read_text(encoding="utf-8")
)
data_audit = pd.DataFrame(
    [
        {
            "Datei": relative,
            "SHA256 damals": digest,
            "SHA256 aktuell": hashlib.sha256((PROJECT_ROOT / relative).read_bytes()).hexdigest(),
        }
        for relative, digest in search_manifest["data"].items()
    ]
)
data_audit["identisch"] = data_audit["SHA256 damals"].eq(data_audit["SHA256 aktuell"])
display(data_audit)
assert data_audit["identisch"].all(), "Daten-/Split-Artefakte unterscheiden sich vom Originallauf."

split = load_split()
folds = load_folds()
reference_X = pd.read_parquet(bundle / "reference" / "features.parquet")
reference_y = pd.read_parquet(bundle / "reference" / "targets.parquet")
reference_predictions = pd.read_parquet(bundle / "reference" / "predictions.parquet")
pd.testing.assert_frame_equal(split.X_val, reference_X)
pd.testing.assert_frame_equal(split.y_val, reference_y)
assert split.X_train.index.is_unique and split.X_val.index.is_unique
assert split.X_train.index.intersection(split.X_val.index).empty
assert split.X_train.columns.equals(split.X_val.columns)
assert not set(TARGET_COLS).intersection(split.X_train.columns)
for fold in folds:
    assert not set(fold.train).intersection(fold.val)
    assert set(fold.train).union(fold.val) == set(range(len(split.X_train)))

train_rows = pd.concat([split.X_train, split.y_train], axis=1)
validation_rows = pd.concat([split.X_val, split.y_val], axis=1)
train_hashes = pd.util.hash_pandas_object(train_rows, index=False)
validation_hashes = pd.util.hash_pandas_object(validation_rows, index=False)
assert not set(train_hashes).intersection(validation_hashes), (
    "Identische gelabelte Zeilen in beiden Teilen."
)
train_context = (
    split.X_train[AEZ_COL].astype(str) + AEZ_MONTH_SEP + split.X_train[MONTH_COL].astype(str)
)
validation_context = (
    split.X_val[AEZ_COL].astype(str) + AEZ_MONTH_SEP + split.X_val[MONTH_COL].astype(str)
)
shared_context = set(train_context).intersection(validation_context)
display(
    pd.Series(
        {
            "Seed": SEED,
            "Validierungsanteil": load_split_config().validation_size,
            "Trainingszeilen": len(split.X_train),
            "Holdout-Zeilen": len(split.X_val),
            "Input-Features": split.X_train.shape[1],
            "CV-Folds": len(folds),
            "Gemeinsame AEZ×Monat-Kontexte": len(shared_context),
        },
        name="Datenprotokoll",
    )
)
display(
    pd.concat(
        {
            "Training": combined_label(split.y_train).value_counts(),
            "Holdout": combined_label(split.y_val).value_counts(),
        },
        axis=1,
    ).sort_index()
)

## 4. Was bedeutet „BAcc kombiniert“?

Im Projekt ist das der ungewichtete Recall-Mittelwert über die **23 gemeinsamen Klassen `Crop|Stage`**, berechnet mit `sklearn.metrics.balanced_accuracy_score`. Es ist **nicht** der Mittelwert aus Kultur-BAcc und Stadium-BAcc und nicht die zeilenweise Trefferquote.

Der folgende Gegencheck berechnet die Kennzahlen direkt mit sklearn. Ein kleiner Testfall mit richtiger Kultur, aber vertauschten Stadien zeigt zusätzlich, warum der Mittelwert der beiden Aufgaben eine andere Kennzahl ist.

In [ ]:
import numpy as np
from sklearn.metrics import balanced_accuracy_score

from awp2.config import CROP_COL, SCORE_DECIMALS, STAGE_COL
from awp2.data import valid_combinations
from awp2.evaluation import evaluate

historical_metrics = evaluate(reference_y, reference_predictions, valid_combinations(split.y_train))
assert historical_metrics.model_dump() == frozen_record["metrics"]
direct_combined = balanced_accuracy_score(
    combined_label(reference_y), combined_label(reference_predictions)
)
direct_crop = balanced_accuracy_score(reference_y[CROP_COL], reference_predictions[CROP_COL])
direct_stage = balanced_accuracy_score(reference_y[STAGE_COL], reference_predictions[STAGE_COL])
assert round(direct_combined, SCORE_DECIMALS) == historical_metrics.bacc_combined
display(
    pd.Series(
        {
            "BAcc kombiniert (sklearn, ungerundet)": direct_combined,
            "BAcc kombiniert (Projekt)": historical_metrics.bacc_combined,
            "Mittelwert aus Kultur-/Stadium-BAcc (andere Metrik)": (direct_crop + direct_stage) / 2,
            "Paar-Trefferquote (andere Metrik)": reference_y.eq(reference_predictions)
            .all(axis=1)
            .mean(),
        },
        name="Metrikprüfung",
    )
)

toy_true = (
    reference_y.loc[reference_y[CROP_COL].eq(reference_y[CROP_COL].iloc[0])]
    .drop_duplicates()
    .head(2)
    .reset_index(drop=True)
)
toy_pred = toy_true.copy()
toy_pred[STAGE_COL] = toy_pred[STAGE_COL].iloc[::-1].to_numpy()
toy_scores = [balanced_accuracy_score(toy_true[col], toy_pred[col]) for col in TARGET_COLS]
toy_combined = balanced_accuracy_score(combined_label(toy_true), combined_label(toy_pred))
assert toy_combined == 0 and np.mean(toy_scores) == 0.5
print(f"Testfall: gemeinsame BAcc={toy_combined}, Aufgabenmittel={np.mean(toy_scores)}")

## 5. Gewinnerkonfiguration rekonstruieren

Die Parameter stammen direkt aus `provenance/validation.json`: **RBF-SVM, C=100, gamma=scale, ausgewogene Klassengewichte**, MinMax-Skalierung, Savitzky-Golay 11/3, Albedo, Vegetationsindizes, AEZ×Monat und zyklischer Monat. Keine PCA, kein SNV.

Die SVM wird ausdrücklich neu konstruiert. Ihre vollständigen Konstruktorparameter werden gegen das gespeicherte Modell geprüft. Die Vorverarbeitung entsteht ausschließlich über `PreprocessingConfig` und die gemeinsame Pipeline in `tune()` / `run()`; keine Transformation wird außerhalb der Trainingsdaten gefittet. Quellcode-Abweichungen gegenüber dem Archiv werden offengelegt, nicht durch einen Branchwechsel versteckt.

In [ ]:
import mlflow.sklearn
from sklearn.svm import SVC

from awp2.models.combined import CombinedLabelClassifier
from awp2.preprocessing import PreprocessingConfig

winner = frozen_record["winner"]
assert winner["name"] == "combined_svm"
preprocessing = PreprocessingConfig.model_validate(winner["preprocessing"])
base_model = CombinedLabelClassifier(SVC(class_weight="balanced", random_state=SEED))
base_model.set_params(**winner["params"])
stored_pipeline = mlflow.sklearn.load_model(str(bundle / "pipeline"))
assert (
    base_model.estimator.get_params() == stored_pipeline.named_steps["model"].estimator.get_params()
)
assert list(split.X_train.columns) == [item["name"] for item in bundle_metadata["input_schema"]]
changed_source = [
    relative
    for relative, digest in search_manifest["source"].items()
    if not (PROJECT_ROOT / relative).is_file()
    or hashlib.sha256((PROJECT_ROOT / relative).read_bytes()).hexdigest() != digest
]
display(pd.Series(winner["params"], name="Feste SVM-Parameter"))
display(pd.Series(winner["preprocessing"], name="Archivierte Vorverarbeitung"))
print(f"{len(changed_source)} Quelldateien weichen vom Archiv ab oder fehlen.")
print("Die nächste Zelle prüft daher ausdrücklich einen frischen Fit mit dem aktuellen Code.")
base_model

## 6. Frischer Fit und identische Auswertung

`tune()` bekommt **genau eine festgelegte Konfiguration**, um deren fünf vorhandene Trainingsfolds zu reproduzieren. Es wird kein neuer Suchraum geöffnet und kein Parameter anhand eines Ergebnisses verändert. Danach fittet `run()` die gesamte Pipeline ausschließlich auf dem Trainingsteil und wertet denselben bekannten Holdout aus. Beide Schritte werden getrennt im MLflow-Experiment `automl-verification` protokolliert.

Die Prüfung erwartet dieselbe CV-BAcc bis zur dokumentierten Metrikrundung sowie identische Holdout-Vorhersagen und sämtliche acht Metriken. Wiederholtes Ausführen dieser Zelle nutzt das Ergebnis im laufenden Kernel erneut; ein Kernel-Neustart führt die Reproduktion nochmals aus.

In [ ]:
from awp2.experiment import RunConfig, TuneConfig, run, tune

verification_name = f"automl_reproduce_{frozen_record['run_id'][:8]}"
verification_options = {
    "approach": "combined",
    "experiment": "automl-verification",
    "description": "Fixed archived AutoML configuration; reproduction, no model selection.",
    "preprocessing": preprocessing,
    "balance_samples": winner["balance_samples"],
}
if "reproduction" not in globals():
    reproduced_tuning = tune(
        base_model,
        TuneConfig(
            name=f"{verification_name}_cv",
            param_grid={name: [value] for name, value in winner["params"].items()},
            **verification_options,
        ),
    )
    reproduction = run(
        reproduced_tuning.best_model,
        RunConfig(
            name=verification_name,
            tuning_run=reproduced_tuning.run_id,
            **verification_options,
        ),
    )

pd.testing.assert_frame_equal(reproduction.y_val, reference_y)
pd.testing.assert_frame_equal(reproduction.y_pred, reference_predictions, check_dtype=False)
assert reproduction.metrics.model_dump() == frozen_record["metrics"]
assert np.isclose(
    reproduced_tuning.best_score,
    winner["score"],
    rtol=0,
    atol=10 ** (-SCORE_DECIMALS),
), "Der frische CV-Score weicht von der archivierten Metrik ab."
score_check = pd.DataFrame(
    {
        "Archiv": [winner["score"], frozen_record["metrics"]["bacc_combined"]],
        "Frisch reproduziert": [reproduced_tuning.best_score, reproduction.metrics.bacc_combined],
    },
    index=["CV-BAcc kombiniert", "Holdout-BAcc kombiniert"],
)
display(score_check)
display(reproduced_tuning.candidates)
print(f"Tuning-Run: {reproduced_tuning.run_id}")
print(f"Reproduktions-Run: {reproduction.run_id}")
print("Frischer Fit: alle Holdout-Vorhersagen und acht Metriken stimmen exakt überein.")

## 7. Gegenüberstellung mit `exp/automl-optuna`

Die Teamwerte werden aus dem tatsächlich vorhandenen Remote-Branch gelesen und dessen Commit wird festgehalten. Sie sind gerundete **Dokumentationswerte**, keine in diesem Notebook neu erzeugten Optuna-Ergebnisse. Das Team beschreibt drei unterschiedliche Suchen und unterschiedliche Gewinner.

Die erste Optuna-Suche hat **0,867 CV-BAcc und 0,879 Holdout-BAcc**. Die erweiterte Suche hat **0,866 CV-BAcc ohne Holdout-Lauf**. Unsere 0,8832 sind ein **Holdout-Wert**, während unsere eigene CV ebenfalls nur 0,86682 erreicht. Ein Vergleich von 0,86 CV mit 0,8832 Holdout wäre daher irreführend.

Die genaue Optuna-Studie und deren per-Sample-Vorhersagen liegen hier nicht als gemeinsame Referenz vor. Aus gerundeten Parametern wird deshalb kein vermeintlich exakter Optuna-Nachbau erzeugt. Ein fairer gepaarter Überlegenheitsnachweis ist mit diesen Angaben nicht möglich.

In [ ]:
import re

optuna_ref = "origin/exp/automl-optuna"
optuna_commit = subprocess.check_output(
    ["git", "rev-parse", optuna_ref], cwd=PROJECT_ROOT, encoding="utf-8"
).strip()
optuna_document = subprocess.check_output(
    ["git", "show", f"{optuna_commit}:docs/modelle/ansaetze/automl.md"],
    cwd=PROJECT_ROOT,
    encoding="utf-8",
)
comparison_rows = [
    {
        "Ansatz": "Fokussiertes Raster: Archiv",
        "CV-BAcc kombiniert": winner["score"],
        "Holdout-BAcc kombiniert": frozen_record["metrics"]["bacc_combined"],
        "Nachweis": "Originalartefakte und sämtliche Vorhersagen geprüft",
    },
    {
        "Ansatz": "Fokussiertes Raster: frischer Fit",
        "CV-BAcc kombiniert": reproduced_tuning.best_score,
        "Holdout-BAcc kombiniert": reproduction.metrics.bacc_combined,
        "Nachweis": "Aktueller Code; identische Daten, Splits und Vorhersagen",
    },
]
for number, title in enumerate(("Optuna kombiniert", "Optuna Aufbau", "Optuna erweitert"), start=1):
    heading = f"### Suche {number}:"
    section = optuna_document.split(heading, maxsplit=1)[1].split("\n### ", maxsplit=1)[0]
    cv_match = re.search(r"CV \*\*(\d+\.\d+)\*\*", section)
    holdout_match = re.search(r"kombiniert \*\*(\d+\.\d+)\*\*", section)
    assert cv_match, f"CV-Wert in {heading} nicht eindeutig gefunden."
    comparison_rows.append(
        {
            "Ansatz": title,
            "CV-BAcc kombiniert": float(cv_match.group(1)),
            "Holdout-BAcc kombiniert": float(holdout_match.group(1)) if holdout_match else np.nan,
            "Nachweis": "Branch-Dokumentation, gerundet; hier nicht neu gefittet",
        }
    )
comparison = pd.DataFrame(comparison_rows).set_index("Ansatz")
display(comparison)
print(f"Quelle: {optuna_ref} @ {optuna_commit}")
print("Optuna-Suchdatenbank lokal vorhanden:", (MODELS_DIR / "automl.db").is_file())
print(
    "Holdout-Abstand zur dokumentierten ersten Optuna-Suche:",
    reproduction.metrics.bacc_combined
    - comparison.loc["Optuna kombiniert", "Holdout-BAcc kombiniert"],
)
display(
    pd.DataFrame(
        {
            "Fokussiertes Raster": [
                "384 feste Kandidaten; gleiches Budget pro Modellraum",
                "C=100, gamma=scale",
                "MinMax; Savitzky-Golay 11/3; Albedo; Vegetationsindizes",
                "Keine PCA; AEZ×Monat; zyklischer Monat",
            ],
            "Optuna Suche 1 (Dokumentation)": [
                "TPE; 542 Versuche, davon 317 abgeschlossen",
                "C≈138, gamma≈0.0021 (gerundet)",
                "Standardisierung",
                "PCA 80; AEZ×Monat; zyklischer Monat",
            ],
        },
        index=["Suchverfahren/Budget", "SVM", "Spektrale Vorverarbeitung", "Weitere Merkmale"],
    )
)

## 8. Fehlerbilder und Prüfbericht sichern

Die niedrigste Paarklassen-Erkennung bleibt `soybean|Emerge_VEarly` mit 0,5714 Recall bei 21 Holdout-Zeilen. Einzelne perfekte Recalls beruhen auf nur zwei bis acht Beispielen und sind entsprechend unsicher.

Konfusionsmatrix und Paarklassenrecall entstehen aus den frisch reproduzierten Vorhersagen. Der Prüfbericht enthält Git-Stand, Daten-Hashes, Konfiguration, CV-Folds, MLflow-IDs und den Optuna-Quellcommit. Zeilenweise Vorhersagen und Tabellen bleiben unter dem ignorierten Scratch-Verzeichnis `reports/figures/`; nur die ausgewählten Abbildungen sind für die Dokumentation vorgesehen.

In [ ]:
import matplotlib.pyplot as plt

from awp2.config import FIGURES_DIR, MODEL_DOCS_FIGURES_DIR
from awp2.evaluation import plot_confusion_matrices, recall_by_class
from awp2.plots import plot_recall_comparison, save_doc_figure

all_metrics = pd.DataFrame(
    {
        "Archiv": frozen_record["metrics"],
        "Frischer Fit": reproduction.metrics.model_dump(),
    }
)
class_recall = recall_by_class(reproduction.y_val, reproduction.y_pred)
display(all_metrics)
display(class_recall)

confusion_figure = plot_confusion_matrices(reproduction.y_val, reproduction.y_pred)
save_doc_figure(confusion_figure, f"{verification_name}_confusion", MODEL_DOCS_FIGURES_DIR)
plt.show()
plt.close(confusion_figure)
recall_figure = plot_recall_comparison({"Frischer Fit": class_recall})
save_doc_figure(recall_figure, f"{verification_name}_recall", MODEL_DOCS_FIGURES_DIR)
plt.show()
plt.close(recall_figure)

output = FIGURES_DIR / verification_name
output.mkdir(parents=True, exist_ok=True)
all_metrics.to_csv(output / "metrics.csv")
score_check.to_csv(output / "cv_holdout_check.csv")
reproduced_tuning.candidates.to_csv(output / "cv_folds.csv", index=False)
comparison.to_csv(output / "optuna_comparison.csv")
class_recall.to_csv(output / "class_recall.csv")
data_audit.to_csv(output / "data_hashes.csv", index=False)
pd.concat(
    {
        "true": reproduction.y_val,
        "predicted": reproduction.y_pred,
        "archived": reference_predictions,
    },
    axis=1,
).to_parquet(output / "predictions.parquet")
(output / "optuna_source.md").write_text(optuna_document, encoding="utf-8")
verification_report = {
    "git_head": git_head,
    "python": platform.python_version(),
    "original_run_id": frozen_record["run_id"],
    "reproduction_tuning_run_id": reproduced_tuning.run_id,
    "reproduction_run_id": reproduction.run_id,
    "reproduction_model_uri": reproduction.model_uri,
    "optuna_source_commit": optuna_commit,
    "seed": SEED,
    "same_data_hashes": bool(data_audit["identisch"].all()),
    "same_holdout_predictions": True,
    "shared_aez_month_contexts": len(shared_context),
    "cv_original": winner["score"],
    "cv_reproduced": reproduced_tuning.best_score,
    "metrics": reproduction.metrics.model_dump(),
    "preprocessing": preprocessing.model_dump(mode="json"),
    "estimator_params": winner["params"],
    "changed_source_files": changed_source,
    "independent_test": False,
    "optuna_refitted_here": False,
}
(output / "verification.json").write_text(
    json.dumps(verification_report, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)
print(f"Prüfbericht und Vorhersagen: {output}")

## 9. Fazit und Grenzen

**Am 29.10.2026 tatsächlich geprüft:**

- Das Archiv liefert mit dem ursprünglichen Code exakt dieselben 1.118 Vorhersagen.
- Daten-, Split- und Konfigurationsdateien sind SHA-256-identisch; die Holdout-Features, Labels und IDs stimmen vollständig.
- Ein neu konstruiertes Modell mit dem aktuellen Code reproduziert **CV-BAcc 0,86682** auf denselben fünf Fold-Zuweisungen und **Holdout-BAcc 0,8832**. Die neu berechneten Fold-Scores stehen in der CV-Tabelle. Alle Holdout-Vorhersagen und acht Metriken stimmen exakt überein.
- Neue MLflow-IDs: CV `fb3b2035d055468c9a469c601c926211`, Fit/Holdout `18f708b4b6e64d3393d1acd6be766773`. Diese IDs dokumentieren diese Ausführung; nach einem Kernel-Neustart erzeugt ein erneuter Fit andere IDs.
- Optuna-Suche 1 berichtet **CV 0,867 / Holdout 0,879**, Suche 2 **0,861 / 0,870**, Suche 3 **0,866 / kein Holdout**. Die vermeintliche große Differenz von 0,86 zu 0,8832 vermischt unterschiedliche Auswertungen und teils unterschiedliche Suchen.

**Was daraus nicht folgt:**

Die rund 0,0042 höhere Holdout-BAcc gegenüber dem gerundeten ersten Optuna-Wert beweist keine Überlegenheit. Beide Suchen waren adaptiv; der Holdout wurde mehrfach betrachtet. Kleine Paarklassen und 32 überlappende AEZ×Monat-Kontexte begrenzen die Aussage. Das Notebook verwendet keinen unabhängigen gelabelten Testdatensatz und prüft nicht die Übertragung auf ungesehene Aufnahmen.

Mangels identischer Optuna-Zeilenprognosen gibt es hier keinen gepaarten Vergleich oder Bootstrap für eine Modelldifferenz. Es werden auch keine zusätzlichen Wiederholungs-/Gruppen-CVs zum nachträglichen Auswählen eines Gewinners gestartet. Dafür wären ein vorab festgelegtes Protokoll und die vollständigen Teamartefakte erforderlich. **Reproduzierbarkeit bestätigt; Überlegenheit nicht nachgewiesen.**